# Team Challenge SQL
## Parte 2: Diseño e Implementación de Base de Datos de Suministros

### Objetivo
Diseñar un esquema relacional para control de piezas, proveedores y suministros,
crearlo en SQLite desde Python, cargar datos sintéticos y ejecutar consultas de validación.

### Modelo relacional (ERD resumen)
- CATEGORIA (1) -- (N) PIEZA
- PROVEEDOR (1) -- (N) SUMINISTRO
- PIEZA (1) -- (N) SUMINISTRO

Entidades:
- CATEGORIA: cod_categoria PK
- PIEZA: cod_pieza PK, FK cod_categoria
- PROVEEDOR: cod_proveedor PK
- SUMINISTRO: cod_suministro PK, FK cod_proveedor, FK cod_pieza

In [ ]:
import sqlite3
import pandas as pd

ruta_db = './data/suministros.db'
conn = sqlite3.connect(ruta_db)
cur = conn.cursor()
for tabla in ['solution','suministro','proveedor','pieza','categoria']:
    cur.execute(f"DROP TABLE IF EXISTS {tabla};")

# Crear tablas
cur.executescript('''
CREATE TABLE categoria (
    cod_categoria TEXT PRIMARY KEY,
    nombre_categoria TEXT NOT NULL UNIQUE,
    descripcion TEXT,
    fecha_creacion DATE DEFAULT (DATE('now'))
);

CREATE TABLE pieza (
    cod_pieza TEXT PRIMARY KEY,
    nombre_pieza TEXT NOT NULL,
    color TEXT,
    precio REAL NOT NULL CHECK (precio >= 0),
    cod_categoria TEXT NOT NULL,
    fecha_alta DATE DEFAULT (DATE('now')),
    FOREIGN KEY (cod_categoria) REFERENCES categoria(cod_categoria) ON DELETE RESTRICT
);

CREATE TABLE proveedor (
    cod_proveedor TEXT PRIMARY KEY,
    nombre_proveedor TEXT NOT NULL,
    direccion TEXT,
    ciudad TEXT,
    provincia TEXT,
    correo TEXT,
    telefono TEXT,
    activo INTEGER NOT NULL CHECK(activo IN (0,1)) DEFAULT 1,
    fecha_alta DATE DEFAULT (DATE('now'))
);

CREATE TABLE suministro (
    cod_suministro INTEGER PRIMARY KEY AUTOINCREMENT,
    cod_proveedor TEXT NOT NULL,
    cod_pieza TEXT NOT NULL,
    cantidad INTEGER NOT NULL CHECK(cantidad > 0),
    fecha_suministro DATE NOT NULL,
    lote TEXT NOT NULL,
    precio_unitario REAL NOT NULL CHECK(precio_unitario >= 0),
    observaciones TEXT,
    FOREIGN KEY (cod_proveedor) REFERENCES proveedor(cod_proveedor) ON DELETE CASCADE,
    FOREIGN KEY (cod_pieza) REFERENCES pieza(cod_pieza) ON DELETE CASCADE,
    UNIQUE(cod_proveedor, cod_pieza, fecha_suministro, lote)
);

CREATE TABLE solution (id INTEGER PRIMARY KEY, value TEXT);
''')
conn.commit()
print('Tablas creadas correctamente.')

Tablas creadas correctamente.


### Insertar datos sintéticos

In [2]:
categorias = [
    ('C01','Electrónica','Componentes electrónicos comunes'),
    ('C02','Mecánica','Piezas mecánicas'),
    ('C03','Plásticos','Piezas plásticas'),
    ('C04','Tornillería','Tornillos y tuercas')
]

piezas = [
    ('P001','Resistor 10k','azul',0.05,'C01'),
    ('P002','Condensador 100uF','negro',0.15,'C01'),
    ('P003','Motor DC 12V','gris',12.0,'C02'),
    ('P004','Rueda 20cm','negro',4.5,'C02'),
    ('P005','Carcasa ABS','blanco',2.3,'C03'),
    ('P006','Tornillo M4x10','metal',0.02,'C04'),
    ('P007','Tuerca M4','metal',0.01,'C04'),
    ('P008','Soporte plástico','azul',1.2,'C03')
]

proveedores = [
    ('V001','Suministros S.A.','C/ Industria 12','Madrid','Madrid','ventas@suministros.com','910000001',1),
    ('V002','ElectroWorld','Av. Tecnología 45','Barcelona','Barcelona','info@electroworld.com','930000002',1),
    ('V003','Mecánica Global','Pza. Taller 8','Valencia','Valencia','contacto@mecanica.global','960000003',1),
    ('V004','PlastiFix','C/ Moldeo 3','Sevilla','Sevilla','ventas@plastifix.es','954000004',1)
]

suministros = [
    ('V001','P001',1000,'2024-01-03','L0001',0.05),
    ('V001','P002',500,'2024-01-03','L0002',0.15),
    ('V002','P003',150,'2024-01-04','L1001',11.9),
    ('V003','P004',400,'2024-01-05','L2001',4.4),
    ('V004','P005',300,'2024-01-06','L3001',2.3),
    ('V004','P008',350,'2024-01-06','L3002',1.2),
    ('V001','P006',2500,'2024-01-07','L0003',0.02),
    ('V001','P007',2500,'2024-01-07','L0004',0.01)
]

cur.executemany('INSERT INTO categoria VALUES (?,?,?,DATE("now"))', categorias)
cur.executemany('INSERT INTO pieza (cod_pieza,nombre_pieza,color,precio,cod_categoria) VALUES (?,?,?,?,?)', piezas)
cur.executemany('INSERT INTO proveedor (cod_proveedor,nombre_proveedor,direccion,ciudad,provincia,correo,telefono,activo) VALUES (?,?,?,?,?,?,?,?)', proveedores)
cur.executemany('INSERT INTO suministro (cod_proveedor,cod_pieza,cantidad,fecha_suministro,lote,precio_unitario) VALUES (?,?,?,?,?,?)', suministros)
conn.commit()
print('Datos insertados correctamente.')

Datos insertados correctamente.


### Consultas de verificación

In [3]:
print('\nCategorías:')
print(pd.read_sql_query('SELECT * FROM categoria', conn).to_string(index=False))

print('\nPiezas:')
print(pd.read_sql_query('SELECT * FROM pieza', conn).to_string(index=False))

print('\nProveedores:')
print(pd.read_sql_query('SELECT * FROM proveedor', conn).to_string(index=False))

print('\nSuministros:')
print(pd.read_sql_query('SELECT * FROM suministro', conn).to_string(index=False))

print('\nSuministros por proveedor y pieza:')
query = '''
SELECT pr.nombre_proveedor, p.nombre_pieza, s.cantidad, s.fecha_suministro, s.lote, s.precio_unitario
FROM suministro s
JOIN proveedor pr ON s.cod_proveedor = pr.cod_proveedor
JOIN pieza p ON s.cod_pieza = p.cod_pieza
ORDER BY pr.cod_proveedor, p.cod_pieza
'''
print(pd.read_sql_query(query, conn).to_string(index=False))


Categorías:
cod_categoria nombre_categoria                      descripcion fecha_creacion
          C01      Electrónica Componentes electrónicos comunes     2026-03-25
          C02         Mecánica                 Piezas mecánicas     2026-03-25
          C03        Plásticos                 Piezas plásticas     2026-03-25
          C04      Tornillería              Tornillos y tuercas     2026-03-25

Piezas:
cod_pieza      nombre_pieza  color  precio cod_categoria fecha_alta
     P001      Resistor 10k   azul    0.05           C01 2026-03-25
     P002 Condensador 100uF  negro    0.15           C01 2026-03-25
     P003      Motor DC 12V   gris   12.00           C02 2026-03-25
     P004        Rueda 20cm  negro    4.50           C02 2026-03-25
     P005       Carcasa ABS blanco    2.30           C03 2026-03-25
     P006    Tornillo M4x10  metal    0.02           C04 2026-03-25
     P007         Tuerca M4  metal    0.01           C04 2026-03-25
     P008  Soporte plástico   azul    1

### Índices y mejoras (opcional)

In [4]:
cur.execute('CREATE INDEX IF NOT EXISTS idx_suministro_proveedor ON suministro(cod_proveedor);')
cur.execute('CREATE INDEX IF NOT EXISTS idx_suministro_pieza ON suministro(cod_pieza);')
conn.commit()

print('Índices creados correctamente.')
conn.close()

Índices creados correctamente.
